# 01 — Bronze ingestion (OSS)

Read manufacturing demo CSVs from `data/raw/`, stamp Bronze provenance, write local Delta.


In [ ]:
from pathlib import Path
from ambient_pipeline.notebook_bootstrap import ensure_pipeline_on_path, apply_spark_tuning

ROOT = ensure_pipeline_on_path(Path.cwd())
assert ROOT is not None, "Run from ambient-core checkout (lib/ambient_pipeline missing)"
print(f"repo root: {ROOT}")


In [ ]:
from ambient_pipeline.perf import create_local_spark

spark = create_local_spark(app_name="ambient-oss-notebooks", shuffle_partitions=4)
apply_spark_tuning(spark)
print(spark.version)


In [ ]:
from datetime import datetime, timezone
from functools import reduce

from pyspark.sql import functions as F

from ambient_pipeline.provenance import BronzeProvenanceStamper
from ambient_pipeline.storage_paths import resolve_table_path

ORG_ID = "demo-org"
RUN_ID = "notebook-bronze-001"
out_base = str(ROOT / ".lakehouse" / "demo")
bronze_table = resolve_table_path("local", out_base, "demo", "bronze", "raw_uploads")

raw_paths = sorted((ROOT / "data" / "raw").glob("Allmanufacturingds-*.csv"))
frames = []
for path in raw_paths:
    df = spark.read.option("header", True).csv(str(path))
    stamper = BronzeProvenanceStamper(
        run_id=RUN_ID,
        org_id=ORG_ID,
        source_type="csv_upload",
        source_path=str(path),
        ingestion_ts=datetime.now(timezone.utc).isoformat(),
    )
    frames.append(stamper.stamp(df.withColumn("_source_file", F.lit(path.name))))

bronze_df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), frames)
(
    bronze_df.write.format("delta")
    .mode("overwrite")
    .partitionBy("_bronze_org_id")
    .save(bronze_table)
)
print(f"bronze rows={spark.read.format('delta').load(bronze_table).count()} path={bronze_table}")
